# Init

In [13]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
TEST_BASIC_DIR = TESTS_DIR / 'osm basic test'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
FIXED_DIR = DATA_DIR / 'fixed'

logger = tgl.initiate_logger('logger', TEST_BASIC_DIR / 'basic_test.log')

process_state = tgm.load(DATA_DIR / "process_state.json")

In [1]:
df_by_cntr = tgm.load_dirs(CLEANED_DIR)
logger.info(f"Cleaned data for countries: {len(df_by_cntr)}")

[2025-12-26 14:48:21] [INFO] : Cleaned data for countries: 83


In [8]:
id_triplets = pd.concat(df_by_cntr.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[2025-12-25 15:53:27] [INFO] : All dataframes id triplets: 155843


# basic test

### * review

In [60]:
test_res_by_cntr = tgm.load_dirs(TEST_BASIC_DIR)
logger.info(f'Test results found: {len(test_res_by_cntr)}')

[2025-12-25 17:10:55] [INFO] : Test results found: 83


In [3]:
temp = pd.DataFrame(test_res_by_cntr)
temp = temp.T.map(len)
temp['test_tags_total'] = [len(elems) for c,elems in df_by_cntr.items() if c in test_res_by_cntr.keys()]
temp.peek()

,missing_name,test_tags_leak,test_tags_in_country,test_tags_NA_result,test_tags_total
Afghanistan,0,0,24,11,35
Albania,0,0,390,0,390
Algeria,0,0,1,2147,2148
Andorra,0,0,1,0,1
Angola,0,0,1,63,64
Anguilla,0,0,1,0,1
AntiguaAndBarbuda,0,0,1,11,12
Argentina,1,0,885,326,1211
Armenia,0,0,27,0,27
Australia,0,0,1,615,616


### * select missing names and leaks from other countries

In [5]:
missing_names = set()
leaks = set()
for k,v in test_res_by_cntr.items():
    missing_names.update(v['missing_name'])
for k,v in test_res_by_cntr.items():
    leaks.update(v['test_tags_leak'])

logger.info(f"Missing names relations: {len(missing_names)}")
logger.info(f"Leaks relations: {len(leaks)}")
relations_from_test_to_delete = leaks | missing_names
logger.info(f"To delete relations (parents): {len(relations_from_test_to_delete)}")

[2025-12-25 15:50:46] [INFO] : Missing names relations: 23
[2025-12-25 15:50:46] [INFO] : Leaks relations: 20
[2025-12-25 15:50:46] [INFO] : To delete relations (parents): 43


### * from the relations to delete, select the childs in the country that has them as parent

In [12]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[2025-12-25 15:54:06] [INFO] : Childs to delete: 4161


### * dump basic_test_to_delete.json

In [13]:
basic_test_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Basic test to delete relations: {len(basic_test_to_delete)}")

[2025-12-25 15:56:40] [INFO] : Basic test to delete relations: 4199


In [14]:
tgm.dump(TEST_BASIC_DIR / "basic_test_to_delete.pkl", basic_test_to_delete)

# first level test

### * review

In [68]:
first_level_test_res = tgm.load_dirs(TEST_FIRST_LEVEL_DIR)
logger.info(f'First level test results found: {len(first_level_test_res)}')

[2025-12-25 18:32:02] [INFO] : First level test results found: 66


In [69]:
first_level_test_res_by_elem = {k:v for list in first_level_test_res.values() for k,v in list.items()}
print(len(first_level_test_res_by_elem))

1236


In [71]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in first_level_test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                   744
[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]              481
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]      7
[[admin_centre, ok], [label, ok], [place, error], [geom_node, ok], [centroid, ok]]                  2
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                1
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, error], [centroid, ok]]                  1
Name: count, dtype: int64

In [72]:
less = set()
for elem, val in first_level_test_res_by_elem.items():
    if len(val) < 4: less.add(elem)
print(len(less))
{id_to_cntr[id[2]] for id in less}

0


set()

In [74]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         5674
missing     503
error         3
Name: count, dtype: int64

### * select parent entities to delete

In [87]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
first_level_test_res_bool = {}
for country, res in first_level_test_res.items():
    for id, val in res.items():
        true_count = 0
        false_count = 0
        for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
            if val[type]['status'] == 'ok':
                # make a voting weight selection
                if val[type]['result'] is True:
                    true_count += 1
                else:
                    false_count += 1
        if (true_count + false_count) < MIN_TESTS:
            first_level_test_res_bool[id] = 'missing'
        else:
            true_ratio = true_count / ((false_count + true_count))
            # test_res_by_tuple_bool[id] = true_count >= false_count
            first_level_test_res_bool[id] = true_ratio > KEEP_THRESHOLD

logger.info(f"Data types: {tgm.tally([v for k,v in first_level_test_res_bool.items()])}")

[2025-12-25 18:37:25] [INFO] : Data types: Counter({True: 1222, False: 14})


In [88]:
first_level_parents_to_delete = {k for k,v in first_level_test_res_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(first_level_parents_to_delete)}')

[2025-12-25 18:37:56] [INFO] : Relations from test to delete: 14


### * from the relations to delete, select the childs in the country that has them as parent

In [91]:
parents_to_delete = first_level_parents_to_delete
first_level_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    first_level_childs_to_delete.update(to_delete)
    parents_to_delete = to_delete
logger.info(f"Childs to delete: {len(first_level_childs_to_delete)}")

[2025-12-25 18:39:31] [INFO] : Childs to delete: 4905


In [92]:
first_level_to_delete = first_level_parents_to_delete | first_level_childs_to_delete
logger.info(f"Current relations to delete: {len(first_level_to_delete)}")

[2025-12-25 18:39:36] [INFO] : Current relations to delete: 4919


In [95]:
tgm.dump(TEST_FIRST_LEVEL_DIR  / "first_level_test_to_delete.pkl", first_level_to_delete)

# duplicates test

### * review

In [102]:
dups_test_res = tgm.load_dirs(TEST_DUPLICATES_DIR)
logger.info(f'Dups test results found: {len(dups_test_res)}')

[2025-12-25 18:43:18] [INFO] : Dups test results found: 32


In [103]:
dups_test_res_by_elem = {k:v for list in dups_test_res.values() for k,v in list.items()}
print(len(dups_test_res_by_elem))

1946


In [104]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in dups_test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1503
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         208
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               193
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               34
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [105]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         7418
missing    2312
Name: count, dtype: int64

### * select parent entities to delete

In [106]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
dups_test_res_bool = {}
for country, res in dups_test_res.items():
    for id, val in res.items():
        true_count = 0
        false_count = 0
        for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
            if val[type]['status'] == 'ok':
                # make a voting weight selection
                if val[type]['result'] is True:
                    true_count += 1
                else:
                    false_count += 1
        if (true_count + false_count) < MIN_TESTS:
            dups_test_res_bool[id] = 'missing'
        else:
            true_ratio = true_count / ((false_count + true_count))
            # test_res_by_tuple_bool[id] = true_count >= false_count
            dups_test_res_bool[id] = true_ratio > KEEP_THRESHOLD

logger.info(f"Data types: {tgm.tally([v for k,v in dups_test_res_bool.items()])}")

[2025-12-25 18:44:15] [INFO] : Data types: Counter({True: 1707, False: 237, 'missing': 2})


In [107]:
dups_parents_to_delete = {k for k,v in dups_test_res_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(dups_parents_to_delete)}')

[2025-12-25 18:44:54] [INFO] : Relations from test to delete: 237


### * from the relations to delete, select the childs in the country that has them as parent

In [109]:
parents_to_delete = dups_parents_to_delete
dups_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    dups_childs_to_delete.update(to_delete)
    parents_to_delete = to_delete
logger.info(f"Childs to delete: {len(dups_childs_to_delete)}")

[2025-12-25 18:45:46] [INFO] : Childs to delete: 1009


In [110]:
dups_to_delete = dups_parents_to_delete | dups_childs_to_delete
logger.info(f"Dups relations to delete: {len(dups_to_delete)}")

[2025-12-25 18:46:32] [INFO] : Dups relations to delete: 1238


In [111]:
tgm.dump(TEST_DUPLICATES_DIR  / "dups_test_to_delete.pkl", dups_to_delete)

### * add manual review

In [ ]:
to_review_triplets = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065'), ('8309397', '391455', '148838'), ('8309397', '1116270', '148838')]

In [ ]:
tgm.dump(TEST_DUPLICATES_DIR / "to_review_triplets.pkl", to_review_triplets)

In [ ]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [ ]:
dups_manual_to_delete = {('6543282', '6543273', '192756'), ('59190', '59195', '59065')}

In [118]:
tgm.dump(TEST_DUPLICATES_DIR / "dups_manual_to_delete.pkl", dups_manual_to_delete)

In [123]:
dups_joined_to_delete = dups_to_delete | dups_manual_to_delete
logger.info(f"Dups joined relations to delete: {len(dups_joined_to_delete)}")

[2025-12-25 18:52:23] [INFO] : Dups joined relations to delete: 1240


In [124]:
tgm.dump(TEST_DUPLICATES_DIR / "dups_joined_to_delete.pkl", dups_joined_to_delete)

# apply fixes

### * countries without all test completed

In [14]:
wihout_completed_tests = set()
for country in df_by_cntr.keys():
    state = process_state[country]
    for task in ['test_basic', 'test_first_level', 'test_duplicates']:
        if state[task]['status'] not in ['ok', 'missing']:
            wihout_completed_tests.add(country)
logger.info(len(wihout_completed_tests))

[2025-12-26 14:55:53] [INFO] : 1


In [15]:
wihout_completed_tests

{'UnitedStates'}

### * join relations to delete

In [127]:
basic_to_delete = tgm.load(TEST_BASIC_DIR / "basic_test_to_delete.pkl")

In [129]:
basic_to_delete = tgm.load(TEST_BASIC_DIR / "basic_test_to_delete.pkl")
logger.info(len(basic_to_delete))

first_level_to_delete = tgm.load(TEST_FIRST_LEVEL_DIR / "first_level_test_to_delete.pkl")
logger.info(len(first_level_to_delete))

dups_joined_to_delete = tgm.load(TEST_DUPLICATES_DIR / "dups_joined_to_delete.pkl")
logger.info(len(dups_joined_to_delete))

[2025-12-25 18:56:19] [INFO] : 4199
[2025-12-25 18:56:19] [INFO] : 4919
[2025-12-25 18:56:19] [INFO] : 1240


In [130]:
to_delete = basic_to_delete | first_level_to_delete | dups_joined_to_delete
print(len(to_delete))

5577


In [131]:
df_by_cntr_fixed = {}
for k,df in df_by_cntr.items():
    df_by_cntr_fixed[k] = df[~df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(to_delete)]
print(len(df_by_cntr_fixed))

83


In [133]:
id_triplets_new = pd.concat(df_by_cntr_fixed.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets_new)}")

[2025-12-25 18:58:36] [INFO] : All dataframes id triplets: 150266


In [134]:
print(len(id_triplets))
print(len(id_triplets)-len(to_delete))
print(len(id_triplets_new))

155843
150266
150266


### * save

In [139]:
for country, df in df_by_cntr_fixed.items():
    tgm.dump(FIXED_DIR / country / f"{country}_fixed.pkl", df)